[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR/blob/main/colab_dae_h.ipynb)

# Part A: the last DAE search on all the images

`colab_dae_full.ipynb` chose salt and pepper 0.5 with dropout 0.3 on h (val k-NN 0.1058). Two stages from there, every run on all the images for 20 epochs, by **val k-NN top-1** (the best epoch is kept). The decoder stays the mirror of the encoder.

**Stage 1, the size of h and dropout.** Val k-NN still rose with dropout (0, 0.1, 0.2, 0.3: 0.0990, 0.1019, 0.1045, 0.1058), so dropout goes higher. h is 256, 2.1% of the input; in the course autoencoder (MNIST) h is 64: 8.2% of the input and 2.0% of the flattened conv output, which here are 1,024 and about 384. h 256, 384, 1,024 × dropout 0.3, 0.4, 0.5: 9 runs, 8 new.

**Stage 2, one change at a time from the winner of stage 1.** Val k-NN peaks after a few epochs and then falls while the reconstruction loss keeps falling: the reconstruction goes on to fine detail that does not help the representation. Each run tries one way to delay that: noise 0.6 and 0.7 (a harder task), rising noise from 0.3 to 0.7 (`--noise_level_end`: harder as the model learns), the L1 loss (`--loss l1`: MSE punishes large errors on single pixels most) and the cosine schedule. 5 runs.

About 12 minutes a run on an L4: 13 new runs, about 2.6 hours. The winner is saved as `"dae"` in `tuning_choice.json`; the earlier choices are kept as `"dae_full20"` (colab_dae_full.ipynb) and `"dae_h"` (stage 1).

## Setup
Downloads the repository (or pulls the latest code), installs it and checks the GPU.

In [1]:
import os

REPO = "/content/Representation-Learning-Midterm-DAE-SimCLR"
if not os.path.exists(REPO):
    !git clone https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR.git {REPO}
os.chdir(REPO)
!git pull
!git log --oneline -1

remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 21 (delta 15), reused 15 (delta 9), pack-reused 0 (from 0)
Unpacking objects: 100% (21/21), 16.43 KiB | 1.49 MiB/s, done.
From https://github.com/NadavFireman/Representation-Learning-Midterm-DAE-SimCLR
   fe4cab6..65b6c22  main       -> origin/main
Updating fe4cab6..65b6c22
Fast-forward
 README.md                                          |    2 +
 colab_dae_h.ipynb                                  |  257 ++++
 config.py                                          |    2 +
 history/colab_dae_full.ipynb                       | 1312 ++++++++++++++++++++
 .../colab_dae_grids.ipynb                          |    0
 .../colab_dae_grids_2.ipynb                        |    0
 model.py                                           |   16 +-
 run_info.py                                        |    9 +-
 utils.py                                           |   

In [2]:
%pip install -q -r requirements.txt

import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → L4 GPU, then rerun this cell."
print(torch.cuda.get_device_name(0), "|", os.cpu_count(), "CPU cores")

NVIDIA L4 | 12 CPU cores


## Drive
Every run writes to `midterm_runs`, next to the earlier runs. The runs here have `tune_pretrain_DAE` in their folder name.

In [3]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_DIR = "/content/drive/MyDrive/לימודים/תואר שני/שנה ב/סמסטר א/למידת ייצוגים במודלים עמוקים/מטלות/מטלת אמצע"
OUT = f"{DRIVE_DIR}/midterm_runs"
assert os.path.isdir(OUT), f"no folder {OUT}"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## TensorBoard
Filter the runs with `tune_pretrain`, then *Toggle all runs*. `epoch/val_knn_top1` shows where each run peaks, `epoch/val_loss` the reconstruction, the TEXT tab each run's `run_info`.

In [4]:
%load_ext tensorboard
%tensorboard --logdir "{OUT}"

Reusing TensorBoard on port 6006 (pid 1916), started 12:47:30 ago. (Use '!kill 1916' to kill it.)

<IPython.core.display.Javascript object>

## The runs
`train` runs train.py only if the run has no `results.json` yet: after a disconnect, run all again; a finished run is read from Drive and an unfinished one continues from its `last.ckpt`.

In [5]:
import importlib
import itertools
import json
import sys

import pandas as pd

RUN = "--mode pretrain --tune --num_epochs 20"  # all the images, the full k-NN bank, val only (no test)
CHOICE_FILE = f"{OUT}/tuning_choice.json"
# The choice of colab_dae_full.ipynb (salt and pepper 0.5, dropout 0.3, val k-NN 0.1058 in 20 epochs)
base = {"noise_type": "salt_pepper", "noise_level": 0.5, "patch_size": 8, "conv1": 32, "conv2": 64,
        "latent_dim": 256, "dropout": 0.3, "augment": "flip", "batch_size": 512, "optimizer": "adamw",
        "activation": "gelu", "lr": 0.001, "wd": 0.01, "scheduler": "none", "noise_type2": "none",
        "noise_level2": 0.0}


def flags_of(params):
    """The train.py flags of one DAE."""
    p = dict(params)
    return f"--channels {p.pop('conv1')} {p.pop('conv2')} " + " ".join(f"--{k} {v}" for k, v in p.items())


def folder_of(flags):
    """The run folder train.py uses for these flags (run_info.folder_name)."""
    import config, utils, run_info
    for module in (config, utils, run_info):
        importlib.reload(module)
    sys.argv = ["train.py", *flags.split(), "--output_dir", OUT]
    return f"{OUT}/{run_info.folder_name(utils.parse_args().mode)}"


def train(flags):
    """One run of train.py (skipped if it finished before): its val numbers and its folder."""
    if not os.path.exists(f"{folder_of(flags)}/results.json"):
        !python train.py {flags} --output_dir "{OUT}"
    folder = folder_of(flags)
    with open(f"{folder}/results.json") as f:
        return {**json.load(f), "run": os.path.basename(folder)}

## Stage 1: h × dropout
Every combination, the rest as in `base`: 3 × 3 = 9 runs. h 256 with dropout 0.3 is the choice of `colab_dae_full.ipynb`, read from Drive.

In [6]:
rows = []
for latent_dim, dropout in itertools.product([256, 384, 1024], [0.3, 0.4, 0.5]):
    params = {**base, "latent_dim": latent_dim, "dropout": dropout}
    r = train(f"{RUN} {flags_of(params)}")
    rows.append({**params, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
stage1 = pd.DataFrame(rows).sort_values("val_knn", ascending=False)
display(stage1[["latent_dim", "dropout", "val_knn", "val_loss", "run"]].round(4))

choices = json.load(open(CHOICE_FILE))
choices.setdefault("dae_full20", choices["dae"])  # the choice of colab_dae_full.ipynb, kept
best1 = {k: stage1.iloc[0][k] for k in base}
best1 = {k: (v.item() if hasattr(v, "item") else v) for k, v in best1.items()}  # numpy -> python, for json
choices["dae_h"] = {**best1, "num_epochs": 20}
with open(CHOICE_FILE, "w") as f:
    json.dump(choices, f, indent=1)
print("stage 1:", best1)

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..
DenoisingAutoEncoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=16384, out_features=256, b

,latent_dim,dropout,val_knn,val_loss,run
1,256,0.4,0.1064,0.2525,date24112026_time130158_CNN3_tune_pretrain_DAE...
0,256,0.3,0.1058,0.2289,date24112026_time112843_CNN3_tune_pretrain_DAE...
4,384,0.4,0.1042,0.2885,date24112026_time133559_CNN3_tune_pretrain_DAE...
3,384,0.3,0.1035,0.2787,date24112026_time132425_CNN3_tune_pretrain_DAE...
5,384,0.5,0.1022,0.2555,date24112026_time134730_CNN3_tune_pretrain_DAE...
6,1024,0.3,0.1006,0.2784,date24112026_time135912_CNN3_tune_pretrain_DAE...
2,256,0.5,0.1005,0.3083,date24112026_time131314_CNN3_tune_pretrain_DAE...
7,1024,0.4,0.0992,0.2817,date24112026_time141212_CNN3_tune_pretrain_DAE...
8,1024,0.5,0.0988,0.2874,date24112026_time142549_CNN3_tune_pretrain_DAE...


stage 1: {'noise_type': 'salt_pepper', 'noise_level': 0.5, 'patch_size': 8, 'conv1': 32, 'conv2': 64, 'latent_dim': 256, 'dropout': 0.4, 'augment': 'flip', 'batch_size': 512, 'optimizer': 'adamw', 'activation': 'gelu', 'lr': 0.001, 'wd': 0.01, 'scheduler': 'none', 'noise_type2': 'none', 'noise_level2': 0.0}


## Stage 2: one change at a time
Each run is the winner of stage 1 (`best1`) with one change; `best1` itself was trained in stage 1 and is read from Drive.

In [7]:
changes = {"stage 1 winner": {}, "noise 0.6": {"noise_level": 0.6}, "noise 0.7": {"noise_level": 0.7},
           "rising noise 0.3 to 0.7": {"noise_level": 0.3, "noise_level_end": 0.7},
           "L1 loss": {"loss": "l1"}, "cosine schedule": {"scheduler": "cosine"}}
rows = []
for name, change in changes.items():
    params = {**best1, **change}
    r = train(f"{RUN} {flags_of(params)}")
    rows.append({"change": name, "params": params, "val_knn": r["val_knn_top1"], "val_loss": r["val_loss"], "run": r["run"]})
stage2 = pd.DataFrame(rows).sort_values("val_knn", ascending=False)
display(stage2[["change", "val_knn", "val_loss", "run"]].round(4))

Seed set to 42
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
`Trainer(limit_train_batches=1.0)` was configured so 100% of the batches per epoch will be used..
`Trainer(limit_val_batches=1.0)` was configured so 100% of the batches will be used..
`Trainer(limit_test_batches=1.0)` was configured so 100% of the batches will be used..
DenoisingAutoEncoder(
  (encoder): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (1): GELU(approximate='none')
    (2): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    (3): GELU(approximate='none')
    (4): Flatten(start_dim=1, end_dim=-1)
    (5): Linear(in_features=16384, out_features=256, b

,change,val_knn,val_loss,run
0,stage 1 winner,0.1064,0.2525,date24112026_time130158_CNN3_tune_pretrain_DAE...
5,cosine schedule,0.1053,0.2478,date24112026_time152446_CNN3_tune_pretrain_DAE...
3,rising noise 0.3 to 0.7,0.1014,0.3393,date24112026_time150206_CNN3_tune_pretrain_DAE...
1,noise 0.6,0.1004,0.3068,date24112026_time143930_CNN3_tune_pretrain_DAE...
4,L1 loss,0.0940,0.3664,date24112026_time151321_CNN3_tune_pretrain_DAE...
2,noise 0.7,0.0929,0.3445,date24112026_time145052_CNN3_tune_pretrain_DAE...


## The choice
The best of stage 2 (the winner of stage 1 if no change beats it) is saved as `"dae"`.

In [8]:
best = stage2.iloc[0]
choices["dae"] = {**best["params"], "num_epochs": 20}
with open(CHOICE_FILE, "w") as f:
    json.dump(choices, f, indent=1)
print(best["change"], "->", choices["dae"])

stage 1 winner -> {'noise_type': 'salt_pepper', 'noise_level': 0.5, 'patch_size': 8, 'conv1': 32, 'conv2': 64, 'latent_dim': 256, 'dropout': 0.4, 'augment': 'flip', 'batch_size': 512, 'optimizer': 'adamw', 'activation': 'gelu', 'lr': 0.001, 'wd': 0.01, 'scheduler': 'none', 'noise_type2': 'none', 'noise_level2': 0.0, 'num_epochs': 20}
